In [135]:
%pip install pandas 


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [136]:
import glob
import pandas as pd

In [137]:
files = sorted(glob.glob("data/stocks/*.csv"))
print(f"{len(files)} files found")

250 files found


In [138]:
df = pd.concat(
    (
        pd.read_csv(f, parse_dates=["window_start"]).assign(date=f[-14:-4])
        for f in files
    ),
    ignore_index=True,
)

df["date"] = pd.to_datetime(df["date"])
df = df.set_index(["date", "ticker"])
df.sort_index(inplace=True)

print(df.shape)
df.head()

(2814320, 7)


volume     open    close      high         low  \
date       ticker                                                    
2025-01-02 A        953587  135.210  133.430  135.7300  132.870000   
           AA      2703886   38.165   37.990   39.0400   37.900000   
           AAA       13720   25.130   25.100   25.1390   25.060100   
           AAAU    2130005   26.160   26.285   26.3000   26.149000   
           AACG      17613    0.832    0.880    0.9054    0.820001   

                          window_start  transactions  
date       ticker                                     
2025-01-02 A       1735794000000000000         20240  
           AA      1735794000000000000         33119  
           AAA     1735794000000000000            71  
           AAAU    1735794000000000000          3266  
           AACG    1735794000000000000           147

In [139]:
import plotly.express as px

MAG7 = ["AAPL", "MSFT", "GOOGL", "AMZN", "NVDA", "META", "TSLA"]

mag7 = (
    df.loc[df.index.get_level_values("ticker").isin(MAG7), "close"]
    .reset_index()
)

fig1 = px.line(
    mag7,
    x="date",
    y="close",
    color="ticker",
    title="Magnificent 7 — Daily Close (2025)",
    labels={"close": "Close Price (USD)", "date": "Date", "ticker": "Stock"},
)
fig1.show()

In [140]:
import plotly.express as px

# Load VIX data
vix_df = pd.read_csv("data/VIX_History.csv", parse_dates=["DATE"])

# Create line chart for VIX CLOSE
fig_vix = px.line(
    vix_df,
    x="DATE",
    y="CLOSE",
    title="VIX Index - Daily Close",
    labels={"CLOSE": "VIX Close", "DATE": "Date"},
)
fig_vix.update_xaxes(rangeslider_visible=True)
fig_vix.show()

In [141]:
from plotly.subplots import make_subplots

dashboard = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=False,
    vertical_spacing=0.08,
    subplot_titles=[
        "Magnificent 7 — Daily Close (2025)",
        "VIX Index — Daily Close Prices (2025)",
        "Interest Rate Influence on Stock Returns",
    ],
)

for trace in fig1.data:
    dashboard.add_trace(trace, row=1, col=1)

for trace in fig_vix.data:
    dashboard.add_trace(trace, row=2, col=1)

for trace in fig_influence.data:
    dashboard.add_trace(trace, row=3, col=1)

dashboard.update_layout(
    height=1000,
    title_text="2025 Stocks, VIX, and Economic Influence Dashboard",
    hovermode="x unified",
)

dashboard.update_xaxes(title_text="Date", row=2, col=1)
dashboard.update_xaxes(title_text="Interest Rate Change (%)", row=3, col=1)
dashboard.update_yaxes(title_text="Close Price (USD)", row=1, col=1)
dashboard.update_yaxes(title_text="VIX Close", row=2, col=1)
dashboard.update_yaxes(title_text="Daily Stock Return (%)", row=3, col=1)

# Temporarily set renderer to browser for dashboard HTML display
import plotly.io as pio
pio.renderers.default = "browser"

dashboard.show()